In [1]:
!pip install -U langchain langchain-community langchain-openai langchain-core

  Using cached langchain-1.2.10-py3-none-any.whl.metadata (5.7 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_openai-1.1.10-py3-none-any.whl.metadata (3.1 kB)
  Using cached langchain_core-1.2.17-py3-none-any.whl.metadata (4.4 kB)
  Using cached langchain_text_splitters-1.1.1-py3-none-any.whl.metadata (3.3 kB)
Using cached langchain-1.2.10-py3-none-any.whl (111 kB)
Using cached langchain_core-1.2.17-py3-none-any.whl (502 kB)
Using cached langchain_community-0.4.1-py3-none-any.whl (2.5 MB)
Using cached langchain_text_splitters-1.1.1-py3-none-any.whl (35 kB)
Using cached langchain_openai-1.1.10-py3-none-any.whl (87 kB)

  Attempting uninstall: langchain-core

    Found existing installation: langchain-core 0.3.83

    Uninstalling langchain-core-0.3.83:

      Successfully uninstalled langchain-core-0.3.83

   ---------------------------------------- 0/5 [langchain-core]
   ---------------------------------------- 0/5 [langchain-c

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-chroma 0.2.2 requires langchain-core!=0.3.0,!=0.3.1,!=0.3.10,!=0.3.11,!=0.3.12,!=0.3.13,!=0.3.14,!=0.3.2,!=0.3.3,!=0.3.4,!=0.3.5,!=0.3.6,!=0.3.7,!=0.3.8,!=0.3.9,<0.4.0,>=0.2.43, but you have langchain-core 1.2.17 which is incompatible.
langchain-postgres 0.0.13 requires langchain-core<0.4.0,>=0.2.13, but you have langchain-core 1.2.17 which is incompatible.


In [2]:
pip install -U langchain langchain-community langchain-openai langchain-core

Note: you may need to restart the kernel to use updated packages.


In [1]:
# Standard Modern Imports
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.runnables import RunnablePassthrough

# FIX: For v0.3.x, these are now found here:
from langchain_core.output_parsers import ResponseSchema, StructuredOutputParser

# If you still require the legacy Router functionality:
from langchain.chains.router import MultiPromptChain
from langchain.chains.router.llm_router import LLMRouterChain, RouterOutputParser

ImportError: cannot import name 'ResponseSchema' from 'langchain_core.output_parsers' (c:\Users\adira\anaconda3\envs\langchainenvjan17\Lib\site-packages\langchain_core\output_parsers\__init__.py)

In [ ]:
from langchain.chains import LLMChain
from langchain.chains.router import MultiPromptChain
from langchain.chains.router.llm_router import LLMRouterChain, RouterOutputParser
from langchain.chains.router.multi_prompt_prompt import MULTI_PROMPT_ROUTER_TEMPLATE
from langchain.output_parsers import ResponseSchema, StructuredOutputParser
from langchain_core.prompts import PromptTemplate
from langchain.chat_models import ChatOpenAI

In [ ]:
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

True

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

C:\Users\adira\AppData\Local\Temp\ipykernel_30592\4162814845.py:1: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the `langchain-openai package and should be used instead. To use it run `pip install -U `langchain-openai` and import as `from `langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)


In [ ]:
#Step1 : Define structured schema for career plan output

plan_schemas = [
    ResponseSchema(name="title", description="The job title the user wants to achieve."),
    ResponseSchema(name="time_frame", description="Time frame to reach the role (e.g., 2 years)."),
    ResponseSchema(name="skills", description="List of skills required.", type="list"),
    ResponseSchema(name="certifications", description="List of recommended certifications.", type="list"),
    ResponseSchema(name="milestones", description="Step-by-step roadmap milestones.", type="list")
]
plan_parser = StructuredOutputParser.from_response_schemas(plan_schemas)
plan_format_instructions = plan_parser.get_format_instructions()

In [ ]:
# Step 2: Function to create advisor prompts with embedded format instructions
def structured_template(domain_description):
    return f"""
You are a career advisor for {domain_description}.
Analyze the user's goal and background. Generate a structured career plan with the following fields:

- title: Desired role
- time_frame: Expected duration
- skills: List of must-have skills
- certifications: List of relevant certifications
- milestones: Step-by-step roadmap

{{format_instructions}}


User Input:
{{input}}
"""


In [ ]:

from langchain.globals import set_verbose, set_debug

set_verbose(True) 
# Or for even more detail:
# set_debug(True)
# Step 3: Define three specialized career advisors
prompt_infos = [
    {"name": "tech", "description": "technical roles like software engineer, cloud architect, data scientist"},
    {"name": "marketing", "description": "roles in product marketing, digital marketing, brand strategy"},
    {"name": "finance", "description": "finance roles like analyst, accountant, or investment banking"}
]

destination_chains = {}
for info in prompt_infos:
    prompt = PromptTemplate(
    template=structured_template(info["description"]),
    input_variables=["input"],
    partial_variables={"format_instructions": plan_format_instructions}
    )

    destination_chains[info["name"]] = LLMChain(llm=llm, prompt=prompt)


In [ ]:
# Step 4: Create the router using MULTI_PROMPT_ROUTER_TEMPLATE
destinations_str = "\n".join([f"{p['name']}: {p['description']}" for p in prompt_infos])
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(destinations=destinations_str)

router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser()
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

In [ ]:
# Step 5: Combine into MultiPromptChain
multi_prompt_chain = MultiPromptChain(
    router_chain=router_chain,
    destination_chains=destination_chains,
    default_chain=destination_chains["tech"],
    verbose=True
)

C:\Users\adira\AppData\Local\Temp\ipykernel_30592\164824675.py:2: LangChainDeprecationWarning: Please see migration guide here for recommended implementation: https://python.langchain.com/docs/versions/migrating_chains/multi_prompt_chain/
  multi_prompt_chain = MultiPromptChain(


In [ ]:
# Step 6: Define final motivational message generator
encouragement_prompt = PromptTemplate(
    input_variables=["input", "title", "time_frame", "skills"],
    template="""
You are a motivational career coach. Based on the following career goal and user plan, write an encouraging summary:

Goal: {input}
Target Role: {title}
Time Frame: {time_frame}
Skills Required: {skills}

Encouragement:"""
)
encouragement_chain = LLMChain(llm=llm, prompt=encouragement_prompt)

In [ ]:
def run_upgraded_career_advisor(user_input):
    # 1. Route the input
    # If this is turn 2, the router needs history to know what "it" means
    route_result = router_chain.invoke({"input": user_input})
    advisor_name = route_result["destination"]

    selected_chain = destination_chains.get(advisor_name, destination_chains["tech"])

    # 2. Generate the career plan
    chain_output = selected_chain.invoke({"input": user_input})
    
    # Ensure chain_output is a string for the parser
    raw_text = chain_output["text"] if isinstance(chain_output, dict) else chain_output
    structured_plan = plan_parser.parse(raw_text)

    # 3. Generate motivational message
    encouragement = encouragement_chain.invoke({
        "input": user_input,
        "title": structured_plan["title"],
        "time_frame": structured_plan["time_frame"],
        "skills": ", ".join(structured_plan["skills"])
    })["text"]

    # FIX: Ensure the output is a string so the Tracer doesn't throw a ValueError
    return {
        "output": f"Advisor {advisor_name} processed the request.", 
        "advisor": advisor_name,
        "plan": structured_plan,
        "encouragement": encouragement.strip()
    }

In [ ]:
# Step 8: Test the system with a sample input
test_input = "I want to transition into product marketing within 2 years. I have a background in sales."
output = run_upgraded_career_advisor(test_input)



> Entering new LLMRouterChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Given a raw text input to a language model select the model prompt best suited for the input. You will be given the names of the available prompts and a description of what the prompt is best suited for. You may also revise the original input if you think that revising it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}
```

REMEMBER: "destination" MUST be one of the candidate prompt names specified below OR it can be "DEFAULT" if the input is not well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
te

ModuleNotFoundError: No module named 'langchain_core.messages.block_translators.langchain_v0'